In [94]:
import masknmf
import torch
import os
import sys
import matplotlib.pyplot as plt
import sys
import numpy as np
from pathlib import Path

from typing import *
import fastplotlib as fpl
import masknmf
import scipy
import scipy.sparse
import numpy as np
import torch
import os
from functools import partial

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
## This is how you load a masknmf object from hdf5
dmr = masknmf.DemixingResults.from_hdf5("/data/lab/IBL_Alyx/cortexlab/Subjects/SP072/2025-08-26/001/suite2p/plane7/masknmf_output/demixing.hdf5")

/data/home/app2139/masknmf-toolbox/masknmf/utils/_serialization.py:176: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  ans[key] = torch.sparse_coo_tensor(


In [87]:
class Single_Session_PETH_Vis:

    def __init__(self, 
                 demixing_result: masknmf.DemixingResults,
                 eta_data: np.ndarray,
                 event_onset_frame: int,
                 neuron_ordering: np.ndarray | list[int],
                 device: str = 'cuda',
                 reference_ranges: dict | None = None):
        """
        Args:
            demixing_result (masknmf.DemixingResults). Demixing Results object
                demixing_result has an attribute, "c" with shape (num_video_frames, num_neurons), which is used in this vis
            eta_data (np.ndarray): Shape (num_timepoints, num_signals). The event triggered trace for each neuron
            event_onset_frame: The time point at which the event occurs; it should be an index in the open interval (0, num_timepoints) -- see traces param documentation
            neuron_ordering: A 1D np.ndarray indicating an ordering of the neurons. 
                E.g. [9, 3, 0, 1 ...] indicates the signal at index 9 from the demixing_result is the highest ranked, etc.
        """
        self._device = device
        self._demixing_result = demixing_result
        self._demixing_result.to(device)
        self._neuron_ordering = neuron_ordering.astype('int')
        global_to_local_map = np.full(self.demixing_result.c.shape[1], np.nan)
        for count, global_id in enumerate(neuron_ordering):
            global_to_local_map[global_id] = count
        self._global_to_local_map = global_to_local_map

        ##Reorder the event trigger data
        self._eta_data = eta_data[:, self.neuron_ordering]

        ## This is an AC Array that you can mask however you'd like, which uses the same shared tensors under the hood as the demixing_result.ac_array
        self._colorful_ac_array = masknmf.ColorfulACArray.from_flyweight(self.demixing_result.shape[1:],
                                                        self.demixing_result.flyweight)

        self._reference_ranges = dict() if reference_ranges is None else reference_ranges

        if 'time' not in self._reference_ranges:
            self._reference_ranges.update({'time': (0, self._demixing_result.shape[0], 1)})

        subplot_names = ['pmd', 'signals selected', 'background', 'all signal']
        
        extent_vid_trace = {
            'pmd': (0.0, 0.5, 0, 0.5),
            'signals selected': (0.5, 1, 0, 0.5),
            'background': (0.0, 0.5, 0.5, 1),
            'all signal': (0.5, 1, 0.5, 1),
        }

        
        self._ndw_vid = fpl.NDWidget(self.reference_ranges,
                                                  names=subplot_names,
                                                  extents = extent_vid_trace,
                                                  controller_ids=[('pmd', 'signals selected', 'background', 'all signal')],
                                                  size=(800, 800))

        self._reference_index = self._ndw_vid.indices

        dims = ["time", "height", "width"]
        self._pmd_panel = self._ndw_vid['pmd'].add_nd_image(self.demixing_result.pmd_array,
                                                                              dims,
                                                                              spatial_dims=dims[1:],
                                                                              name='pmd')


        self._colorful_ac_array = masknmf.ColorfulACArray.from_flyweight(self.demixing_result.shape[1:],
                                                        self.demixing_result.flyweight)
        self._colorful_ac_array.mask = self._colorful_ac_array.mask * 0 ## No profiles visible to start
        colorful_ac_dims=['time', 'height', 'width', 'color']
        self._colorful_ac_panel = self._ndw_vid['signals selected'].add_nd_image(self._colorful_ac_array,
                                                                        tuple(colorful_ac_dims),
                                                                        spatial_dims=tuple(colorful_ac_dims[1:]),
                                                                        rgb_dim='color',
                                                                        name='selected signals')

        self._background_panel =  self._ndw_vid['background'].add_nd_image(self.demixing_result.fluctuating_background_array,
                                                                              dims,
                                                                              spatial_dims=dims[1:],
                                                                              name='background')

        self._colorful_ac_panel_net = self._ndw_vid['all signal'].add_nd_image(self.demixing_result.colorful_ac_array,
                                                                        tuple(colorful_ac_dims),
                                                                        spatial_dims=tuple(colorful_ac_dims[1:]),
                                                                        rgb_dim='color',
                                                                        name='all signal')
        

        self._ndw_traces = fpl.NDWidget(names=['ETA Trace'],
                                       shape = (1, 1),
                                       size = (1600, 800))

        self._ndw_raster = fpl.NDWidget(#self.reference_ranges,
                                        names=['raster'],
                                        shape=(1,1),
                                        size=(800, 800))

        self._averaged_traces = self.compute_averaged_c()[:, self.neuron_ordering] #Shape (binned_frames, num_signals)
        self._ndw_raster_graphic = self.ndw_raster['raster'].add_nd_image(self._averaged_traces.T,
                                                                ("height", "width"),
                                                                ("height", "width"),
                                                               name = 'trace_raster')

        
        
        self._eta_ndt = self.ndw_traces['ETA Trace'].add_nd_timeseries(fpl.utils.heatmap_to_positions(self.eta.T, np.arange(self.eta.shape[0])),
                                                                   dims=("l", 'ETA Trace', "d"),
                                                                    spatial_dims=("l", 'ETA Trace', "d"),
                                                                   slider_dim_transforms={'ETA Trace': np.arange(self.eta.shape[0])},
                                                                    display_window=None,
                                                                    x_range_mode="auto",
                                                                   name='ETA Trace'
                                                                )

        traces_subplot = self.ndw_traces['ETA Trace'].subplot
        traces_subplot.toolbar = False
        traces_subplot.controller.add_camera(traces_subplot.camera, include_state={"x", "width"})
        
        ## Now add the selectors to coordinate everything properly
        self._selection_vector = fpl.SelectionVector()
         
        # add a selector for the the raster plot
        self._raster_image_selector = fpl.ImageHighlightSelector(
            lut="tab10",
            lut_wrap="repeat",
            selection_options={"rows": [i for i in range(self._averaged_traces.shape[1])]},
            options_color="w",
            options_alpha=0.1,
            alpha=0.95,
        )

        self._raster_image_selector.add_graphic(self._ndw_raster_graphic.graphic)
        self._ndw_raster_graphic.graphic.add_event_handler(partial(self.raster_selection), "double_click")

        ## Add a selector to decide which event-triggered average trace to show
        self._traces_visible_selector = fpl.VisibilitySelector(self._eta_ndt.graphic, lut="tab10", lut_wrap="repeat")

        ## Add a selector to show contours of the components
        # add selectors
        self._ac_contour_image_selector = fpl.ImageHighlightSelector(
            lut="tab10",
            lut_wrap="repeat",
            selection_options={"pixels": self.demixing_result.ac_array.contours},
            options_color="w",
            options_alpha=0.0,
            alpha=0.95,
        )
        

        self._ac_contour_image_selector.add_graphic(self._colorful_ac_panel.graphic)
        self._ac_contour_image_selector.add_graphic(self._pmd_panel.graphic)
        self._ac_contour_image_selector.selection = None
        

        ## Now add these selectors to the selection vector to coordinate selections
        self._selection_vector.add_selector((self._raster_image_selector, self._global_to_local_map))
        self._selection_vector.add_selector((self._traces_visible_selector, self._global_to_local_map))
        self._selection_vector.add_selector((self._ac_contour_image_selector, self._global_to_local_map))

        
        # Turn off tooltip
        for subplot in self.ndw_vid.figure:
            subplot.tooltip.enabled = False

        

        
    def raster_selection(self, ev):
        col, row = ev.pick_info['index']
        self._raster_image_selector.selection = int(row)

        mask = self._colorful_ac_array.mask
        mask *= 0
        mask[row] = 1
        self._colorful_ac_array.mask = mask

        time_val = self.ndw_vid.indices['time']
        self.ndw_vid.indices.set_dim_index('time', time_val)
        for subplot in self.ndw_traces.figure:
            subplot.auto_scale()
        
        
        

    def compute_averaged_c(self, factor: int = 10):
        c = self.demixing_result.c.cpu().numpy()
        c = c[:factor*(c.shape[0]//factor), :]
        c = c.reshape((c.shape[0] // factor, factor, c.shape[1]))
        return c.mean(1)

    @property
    def eta(self):
        return self._eta_data

    @property
    def reference_ranges(self):
        return self._reference_ranges

    @property
    def reference_indices(self):
        return self._reference_index
        
    @property
    def demixing_result(self):
        return self._demixing_result

    @property
    def device(self):
        return self._device

    @property
    def neuron_ordering(self):
        return self._neuron_ordering

    @property
    def ndw_traces(self):
        return self._ndw_traces

    @property
    def ndw_vid(self):
        return self._ndw_vid

    @property
    def ndw_raster(self):
        return self._ndw_raster

    def show(self):
        # parse based on canvas type
        if self.ndw_vid.figure.canvas.__class__.__name__ == "JupyterRenderCanvas":
            from ipywidgets import VBox, HBox
            # return HBox([VBox([self.ndw_vid.show(), self.ndw_traces.show()]), self.ndw_raster.show()])
            return VBox([HBox([self.ndw_vid.show(),self.ndw_raster.show()]), self.ndw_traces.show()])
        else:
            return self.ndw_vid.show(), self.ndw_traces.show(), self.ndw_raster.show()



In [89]:
event_trigger_data = dmr.c[100:500:10, :].cpu().numpy() ## This is a placeholder, you can compute event triggered traces for all neurons however you'd like
neuron_ordering = masknmf.demixing.demixing_utils.brightness_order(dmr.a, dmr.c)[0].cpu().numpy() ## This is just an arbitrary brightness ordering

peth_vis = Single_Session_PETH_Vis(dmr,
                                   event_trigger_data,
                                   10,
                                   neuron_ordering,
                                   device='cuda')

peth_vis.show()

RFBOutputContext()

RFBOutputContext()

RFBOutputContext()

/data/home/app2139/fastplotlib/fastplotlib/widgets/nd_widget/_ndw_subplot.py:88: UserWarning: No reference range specified for non-spatial dim 'ETA Trace', auto-generating an `AutoRangeContinuous(0, 40, 1)`.
  warnings.warn(
